# OpusClip Pipeline — Debug & Inspection Notebook

This notebook runs the full pipeline **step by step**, exposing every intermediate
result for debugging, latency analysis, and failure isolation.

**Important**:
- Set **Runtime -> Change runtime type -> T4 GPU** (recommended)
- Set your **OPUSCLIP_API_KEY** in Colab Secrets or Cell 2
- Each cell is self-contained — run in order

---
## Cell 1 — Install Dependencies

In [ ]:
import subprocess, sys, os, pathlib, json, importlib, time

def _sh(cmd, label=""):
    print(f"  -> {label or cmd[:80]}")
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode != 0 and r.stderr.strip():
        print(f"    ! {r.stderr.strip()[:200]}")
    return r

python = sys.executable
print(f"  Python == {sys.version.split()[0]}")

# --- ffmpeg ---
if _sh("ffmpeg -version", "ffmpeg").returncode != 0:
    _sh("apt-get install -y -q ffmpeg > /dev/null 2>&1", "install ffmpeg")
ffmpeg_ver = subprocess.run(["ffmpeg", "-version"], capture_output=True, text=True).stdout.splitlines()[0]
print(f"  ffmpeg  == {ffmpeg_ver}")

# --- yt-dlp ---
if _sh("yt-dlp --version").returncode != 0:
    _sh(f'"{python}" -m pip install -q yt-dlp', "pip yt-dlp")
ytdlp_ver = subprocess.run(["yt-dlp", "--version"], capture_output=True, text=True).stdout.strip()
print(f"  yt_dlp  == {ytdlp_ver}")

# --- opusclip ---
try:
    import opusclip
    print(f"  opusclip == {opusclip.__version__} (already installed)")
except ModuleNotFoundError:
    _sh(f'"{python}" -m pip install -q git+https://github.com/ABo-EsMaiL/OpusClip.git@v1.0.0',
        "pip opusclip (GitHub)")
    importlib.invalidate_caches()
    import opusclip
print(f"    source  == {opusclip.__file__}")

# --- fonts ---
_sh("apt-get install -y -q fonts-noto-core fonts-noto-extra fonts-dejavu-core > /dev/null 2>&1", "fonts")

# --- torch / CUDA ---
import torch
print(f"  torch   == {torch.__version__}")
print(f"  CUDA    == {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"  GPU     == {torch.cuda.get_device_name(0)}")

print("\nAll dependencies ready.")

---
## Cell 2 — Configuration

Prints every `PipelineConfig` field.

In [ ]:
from opusclip.config import PipelineConfig
from dataclasses import fields

# --- API Key ---
try:
    from google.colab import userdata
    os.environ.setdefault("OPUSCLIP_API_KEY", userdata.get("OPUSCLIP_API_KEY"))
    print("API key loaded from Colab Secrets")
except (ImportError, ValueError, KeyError):
    os.environ.setdefault("OPUSCLIP_API_KEY", "")

os.environ.setdefault("LLM_BASE_URL", "https://opencode.ai/zen/v1")
os.environ.setdefault("LLM_MODEL", "deepseek-v4-flash-free")

if not os.environ.get("OPUSCLIP_API_KEY"):
    raise RuntimeError("OPUSCLIP_API_KEY is not set.")

config = PipelineConfig.from_env()
print(f"PipelineConfig fields ({len(fields(config))}):")
for f in fields(config):
    val = getattr(config, f.name)
    print(f"  {f.name:30s} = {val}")

---
## Cell 3 — Input Selection

YouTube URL or upload. Prints full video metadata.

In [ ]:
import subprocess, os, pathlib

def _sh(cmd, label=""):
    print(f"  -> {label or cmd[:80]}")
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode != 0 and r.stderr.strip():
        print(f"    ! {r.stderr.strip()[:200]}")
    return r

INPUT_MODE  = "youtube"   # "upload"  |  "youtube"
YOUTUBE_URL = "https://youtu.be/OwDU3GH9cGI?si=MeHrfxVtdIUK5u9c"

work_dir = pathlib.Path.cwd()
VIDEO_PATH = None

if INPUT_MODE == "upload":
    from google.colab import files as _cf
    print("Select your video file ...")
    _uploaded = _cf.upload()
    VIDEO_PATH = str(work_dir / list(_uploaded.keys())[0])
elif INPUT_MODE == "youtube":
    dest = work_dir / "source_video.mp4"
    VIDEO_PATH = str(dest)
    if dest.exists():
        print(f"  Cached: {dest.name} ({dest.stat().st_size / 1_048_576:.1f} MB)")
    else:
        print(f"  Downloading: {YOUTUBE_URL}")
        r = _sh(
            'yt-dlp --format "bestvideo[height<=1080][ext=mp4]'
            '+bestaudio[ext=m4a]/best[height<=1080]" '
            '--merge-output-format mp4 --no-playlist '
            f'--output "{VIDEO_PATH}" "{YOUTUBE_URL}"',
        )
        if r.returncode != 0:
            raise RuntimeError(f"Download failed: {r.stderr[-300:]}")

print(f"\n  File  : {VIDEO_PATH}")
print(f"  Size  : {os.path.getsize(VIDEO_PATH) / 1_048_576:.1f} MB")

import cv2
cap = cv2.VideoCapture(VIDEO_PATH)
w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = cap.get(cv2.CAP_PROP_FPS)
nf = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
dur = nf / fps if fps else 0
cap.release()
print(f"  Res   : {w}x{h}")
print(f"  FPS   : {fps:.2f}")
print(f"  Frames: {nf}")
print(f"  Dur   : {dur:.1f}s ({dur/60:.1f} min)")

---
## Cell 4 — Transcription

Audio extraction + Whisper. Every intermediate metric printed.

In [ ]:
import time, traceback

from opusclip.subprocess_utils import run_ffmpeg
from opusclip.transcription.whisper_provider import WhisperProvider

output_dir = pathlib.Path(str(config.output_dir))
output_dir.mkdir(exist_ok=True)

# --- extract audio ---
audio_path = output_dir / "audio.wav"
t0 = time.perf_counter()
run_ffmpeg([
    "-y", "-i", VIDEO_PATH,
    "-vn", "-acodec", "pcm_s16le",
    "-ar", "16000", "-ac", "1",
    str(audio_path),
])
t1 = time.perf_counter()
print(f"  Audio extraction : {t1 - t0:.2f}s")
print(f"  Audio file       : {audio_path}")
print(f"  Audio size       : {audio_path.stat().st_size / 1_048_576:.1f} MB")

# --- transcribe ---
t0 = time.perf_counter()
whisper = WhisperProvider(
    model_size=config.whisper_model,
    device=config.whisper_device,
)
lang_hint = "ar"
try:
    result = whisper.transcribe(audio_path, lang_hint)
    t1 = time.perf_counter()
    print(f"\n  Transcription    : {t1 - t0:.2f}s")
    print(f"  Language         : {result.language}")
    print(f"  Duration         : {result.duration:.1f}s")
    print(f"  Segments         : {len(result.segments)}")
    print(f"  Total words      : {len(result.words)}")
    transcript_text = " ".join(s.text for s in result.segments)
    print(f"  Char count       : {len(transcript_text)}")
    tokens_per_char = 4  # approximate for Arabic
    print(f"  Est. tokens      : {len(transcript_text) // tokens_per_char}")
    print()
    print(f"  First 5 segments:")
    for s in result.segments[:5]:
        print(f"    [{s.start:.1f}-{s.end:.1f}] {s.text[:100]}")
    print(f"  Last 5 segments:")
    for s in result.segments[-5:]:
        print(f"    [{s.start:.1f}-{s.end:.1f}] {s.text[:100]}")

    # --- save ---
    transcript_data = {
        "segments": [(s.id, s.text, s.start, s.end) for s in result.segments],
        "words": [(w.word, w.start, w.end, w.probability) for w in result.words],
        "language": result.language,
        "duration": result.duration,
    }
    transcript_path = output_dir / "transcript.json"
    with open(transcript_path, "w", encoding="utf-8") as f:
        json.dump(transcript_data, f, ensure_ascii=False, indent=2)
    print(f"\n  Saved            : {transcript_path}")

except Exception:
    traceback.print_exc()
    raise

---
## Cell 5 — Transcript Repair

Fills missing words Whisper drops. Shows before/after stats.

In [ ]:
from opusclip.transcription.base import TranscriptResult, TranscriptSegment, WordInfo
from opusclip.transcription.word_repair import fill_missing_words

segments = [
    TranscriptSegment(id=sid, text=stxt, start=sstart, end=send, words=[])
    for sid, stxt, sstart, send in transcript_data["segments"]
]
words = [
    WordInfo(word=wword, start=wstart, end=wend, probability=wprob)
    for wword, wstart, wend, wprob in transcript_data["words"]
]

transcript_obj = TranscriptResult(
    segments=segments,
    words=words,
    language=transcript_data["language"],
    duration=transcript_data["duration"],
)

t0 = time.perf_counter()
repaired = fill_missing_words(transcript_obj)
t1 = time.perf_counter()

print(f"  Repair time      : {t1 - t0:.4f}s")
print(f"  Original words   : {len(words)}")
print(f"  Repaired words   : {len(repaired)}")
print(f"  Inserted words   : {len(repaired) - len(words)}")
print(f"\n  Sample repaired (first 10):")
for w in repaired[:10]:
    marker = " << INSERTED" if w.word.startswith("[") else ""
    print(f"    {w.word:20s} {w.start:.2f}-{w.end:.2f} {w.probability:.3f}{marker}")

# store for pipeline
repaired_words = [(w.word, w.start, w.end, w.probability) for w in repaired]

---
## Cell 6 — Clip Selection Preparation

Builds the prompt. Shows compression stats before the API call.

In [ ]:
from opusclip.clip_selection.prompts import get_clip_selection_prompt
from opusclip.clip_selection.llm_selector import LLMClipSelector
from opusclip.security import load_api_key

# --- build prompt ---
system_prompt = get_clip_selection_prompt(
    min_clips=config.min_clips,
    max_clips=config.max_clips,
    min_duration=config.min_duration,
    max_duration=config.max_duration,
    min_virality=config.min_virality,
    lang_hint=transcript_data["language"] or "ar",
)

full_lines = [f"[{s.start:.1f}s-{s.end:.1f}s]: {s.text}" for s in segments]
full_transcript = "\n".join(full_lines)
full_chars = sum(len(l) for l in full_lines)

print(f"  Full transcript chars   : {full_chars}")
print(f"  System prompt chars     : {len(system_prompt)}")

# --- compress ---
selector = LLMClipSelector(
    api_key=load_api_key(),
    base_url=config.llm_base_url,
    model=config.llm_model,
)
compressed = selector._compress_transcript(segments, config.max_llm_chars)
compressed_chars = len(compressed)
ratio = compressed_chars / full_chars if full_chars else 1.0
print(f"  Compressed chars        : {compressed_chars}")
print(f"  Compression ratio       : {ratio:.1%}")

tokens_per_char = 4
print(f"  Est. prompt tokens      : ~{(len(system_prompt) + compressed_chars) // tokens_per_char}")
print(f"  Est. completion tokens  : ~1500")
print(f"  Est. total context      : ~{(len(system_prompt) + compressed_chars) // tokens_per_char + 1500}")
print(f"  max_llm_chars           : {config.max_llm_chars}")
print(f"  API model               : {config.llm_model}")
print(f"  API base URL            : {config.llm_base_url}")

# --- save ---
prompt_path = output_dir / "llm_prompt.txt"
with open(prompt_path, "w", encoding="utf-8") as f:
    f.write("=== SYSTEM PROMPT ===\n")
    f.write(system_prompt)
    f.write("\n\n=== USER PROMPT (transcript) ===\n")
    f.write(compressed)
print(f"\n  Prompt saved to        : {prompt_path}")

---
## Cell 7 — LLM API Call (Raw)

Makes the raw API call and prints the complete response before any parsing.
Most important cell for debugging API latency or response format issues.

In [ ]:
import traceback

print(f"  Request start time  : {time.strftime('%H:%M:%S')}")
print(f"  Request size (chars): {len(system_prompt) + len(compressed)}")
t0 = time.perf_counter()

try:
    resp = selector.client.chat.completions.create(
        model=config.llm_model,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": f"Transcript:\n\n{compressed}"},
        ],
        temperature=0.20,
    )
    t1 = time.perf_counter()

    print(f"  Request finish time : {time.strftime('%H:%M:%S')}")
    print(f"  API latency         : {t1 - t0:.2f}s")
    print()

    choice = resp.choices[0]
    raw_content = choice.message.content or ""
    print(f"  finish_reason       : {choice.finish_reason}")
    print(f"  prompt length       : {len(system_prompt) + len(compressed)}")
    print(f"  response length     : {len(raw_content)}")
    print()

    # usage
    if hasattr(resp, "usage") and resp.usage:
        print(f"  usage.prompt_tokens     : {resp.usage.prompt_tokens}")
        print(f"  usage.completion_tokens : {resp.usage.completion_tokens}")
        print(f"  usage.total_tokens       : {resp.usage.total_tokens}")
    else:
        print("  usage: not available")
    print()

    print("=" * 60)
    print("RAW LLM RESPONSE")
    print("=" * 60)
    print(raw_content)
    print("=" * 60)

    # --- save to disk ---
    raw_path = output_dir / "llm_raw_response.txt"
    with open(raw_path, "w", encoding="utf-8") as f:
        f.write(raw_content)
    print(f"\n  Raw response saved : {raw_path}")

except Exception:
    t1 = time.perf_counter()
    print(f"  API call failed after {t1 - t0:.2f}s:")
    traceback.print_exc()
    raise

---
## Cell 8 — JSON Parsing & Clip Candidates

Parses the LLM response, validates, and filters by duration/virality.

In [ ]:
from opusclip.utils.json_utils import extract_json_array
from opusclip.clip_selection.base import ClipCandidate
from opusclip.exceptions import ClipSelectionError

required_fields = {
    "clip_number", "title", "start", "end",
    "virality_score", "score_breakdown", "hook_line", "reason",
}

parsed = extract_json_array(raw_content)
if parsed is None:
    raise ClipSelectionError("Failed to extract JSON array from LLM response.")

print(f"  JSON extraction   : success")
print(f"  Raw objects       : {len(parsed)}")

validated = []
validation_errors = 0
duration_discards = 0
virality_discards = 0

for c in parsed:
    if not required_fields.issubset(c.keys()):
        missing = required_fields - c.keys()
        print(f"    FAIL validation (missing {missing}): {c.get('title', '?')[:40]}")
        validation_errors += 1
        continue
    try:
        v_score = float(c["virality_score"])
        c_start = float(c["start"])
        c_end = float(c["end"])
    except (TypeError, ValueError):
        validation_errors += 1
        continue

    if v_score < config.min_virality:
        print(f"    DISCARD virality ({v_score} < {config.min_virality}): {c['title'][:40]}")
        virality_discards += 1
        continue

    duration = round(c_end - c_start, 1)
    if not (config.min_duration <= duration <= config.max_duration):
        print(f"    DISCARD duration ({duration}s): {c['title'][:40]}")
        duration_discards += 1
        continue

    validated.append(ClipCandidate(
        start=c_start, end=c_end, score=v_score,
        title=str(c["title"]), summary=str(c["reason"]),
    ))

print()
print(f"  Validation failures      : {validation_errors}")
print(f"  Discarded (virality)     : {virality_discards}")
print(f"  Discarded (duration)     : {duration_discards}")
print(f"  Clip candidates          : {len(validated)}")
print()

validated.sort(key=lambda x: x.score, reverse=True)
for i, cc in enumerate(validated):
    print(f"  #{cc.clip_number or i+1}: score={cc.score}, {cc.start:.1f}-{cc.end:.1f}s, {cc.title[:60]}")

if not validated:
    raise ClipSelectionError("No valid clips after filtering.")

---
## Cell 9 — Render Subtitles Only

Generates ASS subtitle files for all selected clips. No video rendering.

In [ ]:
from opusclip.subtitle.ass_builder import ASSSubtitleRenderer
from opusclip.subtitle.base import WordTiming
from opusclip.subtitle.text_cleaner import clean_for_subtitle
from opusclip.fonts import FontManager

subs_dir = output_dir / "subtitles"
subs_dir.mkdir(exist_ok=True)

word_timings = [
    WordTiming(word=clean_for_subtitle(w[0]), start=float(w[1]), end=float(w[2]))
    for w in repaired_words
]
print(f"  Word timings      : {len(word_timings)}")

font_manager = FontManager()
renderer = ASSSubtitleRenderer(font_manager=font_manager)

t0 = time.perf_counter()
subtitle_paths = []
for cc in validated:
    num = cc.clip_number or validated.index(cc) + 1
    out_path = subs_dir / f"clip_{num:02d}.ass"
    renderer.render(
        words=word_timings,
        clip_start=cc.start,
        clip_end=cc.end,
        config=config,
        output_path=out_path,
    )
    sz = out_path.stat().st_size
    print(f"  Subtitle #{num}: {out_path.name} ({sz} bytes, {cc.start:.1f}-{cc.end:.1f}s)")
    subtitle_paths.append((num, out_path))
t1 = time.perf_counter()
print(f"\n  Total subtitle time: {t1 - t0:.3f}s")

---
## Cell 10 — Render One Clip

Renders only the **first valid clip** so debugging is fast.

In [ ]:
from opusclip.context import PipelineContext
from opusclip.rendering.ffmpeg_optimized_renderer import FFmpegOptimizedRenderer
from opusclip.face_detection.mediapipe_detector import MediaPipeFaceDetector
from opusclip.rendering.validator import validate_rendered_video
import traceback

cc = validated[0]
num = cc.clip_number or 1
sub_map = {n: p for n, p in subtitle_paths}
sub_path = sub_map.get(num)

clips_dir = output_dir / "clips"
clips_dir.mkdir(exist_ok=True)

ctx = PipelineContext(
    video_path=pathlib.Path(VIDEO_PATH),
    video_width=w, video_height=h,
    video_fps=fps,
    target_width=1080, target_height=1920,
    output_dir=output_dir,
    metadata_output_dir=output_dir / "metadata",
)
ctx.src_crop_w = int(h * (1080 / 1920) / (w / h))
if ctx.src_crop_w % 2:
    ctx.src_crop_w += 1
if ctx.src_crop_w > w or ctx.src_crop_w == 0:
    ctx.src_crop_w = w

print(f"  Rendering backend : {config.renderer_backend}")
print(f"  Encoder           : {config.encoder}")
print(f"  Clip              : #{num} ({cc.start:.1f}-{cc.end:.1f}s)")
print(f"  Subtitle file     : {sub_path}")
print(f"  Video path        : {VIDEO_PATH}")
print(f"  Crop width        : {ctx.src_crop_w}")
print()

face_detector = MediaPipeFaceDetector(
    model_asset_path=config.mediapipe_model_path,
)
video_renderer = FFmpegOptimizedRenderer(
    face_detector=face_detector,
    config=config,
)

t0 = time.perf_counter()
try:
    rendered = video_renderer.render_clip(ctx, cc, sub_path)
    t1 = time.perf_counter()
    print(f"  Render time       : {t1 - t0:.2f}s")
    print(f"  Output path       : {rendered.path}")
    if rendered.path.exists():
        print(f"  Output size       : {rendered.path.stat().st_size / 1_048_576:.1f} MB")
    print(f"  Thumbnail path    : {rendered.thumbnail_path}")

    print(f"\n  Validating output ...")
    try:
        validate_rendered_video(rendered.path, 1080, 1920)
        print(f"  Validation        : PASS")
    except Exception as ve:
        print(f"  Validation        : FAIL - {ve}")

except Exception:
    t1 = time.perf_counter()
    print(f"  FAILED after {t1 - t0:.2f}s:")
    traceback.print_exc()

print(f"\n  Debug artifacts in: {output_dir.resolve()}")

---
## Summary of Saved Artifacts

| File | Cell | Description |
|------|------|-------------|
| `opusclip_output/transcript.json` | 4 | Full transcription with words and segments |
| `opusclip_output/llm_prompt.txt` | 6 | Exact prompt sent to the LLM |
| `opusclip_output/llm_raw_response.txt` | 7 | Raw LLM response before parsing |
| `opusclip_output/subtitles/clip_*.ass` | 9 | Generated subtitle files |
| `opusclip_output/clips/clip_*.mp4` | 10 | Rendered video clips |

---
*All times use `time.perf_counter()` for sub-millisecond precision.*